# LAB 4 : document classification by fine-tuning a pretrained language model

In this lab, we train a document classifier using a pretrained language model to get the vectorial representation of the input document.

We will use the French "flaubert model" available on HuggingFace.
It was trained on a masked language modeling task.

The forward propagation of the classifier network is
- forward propagation of flaubert
- keep the contextual embedding of the first token of each sequence (the special token)
- apply a one-hidden layer MLP on top of it

A simple option can be used to either:
- freeze the flaubert's parameters : the pre-trained language model is only used as feature extractor, to obtain a document representation
- let the flaubert's parameters be updated while learning on the downstream task of document classification (= "fine-tuning" the PLM parameters)

## Enable use of a GPU

Using large language models practically requires to use a GPU, computation being too slow on a CPU.

In order to use a GPU:
(TODO only at the end, when actually testing / training on full datasets)
- \> Run / Exécution
- \> Modify execution type / modifier le type d'exécution
- \> Hardware accelerator / accélérateur matériel
- => select "T4 GPU"

**NB** : when using a GPU device, all the input tensors that will be used in computation on the GPU need to be explicitely copied on the GPU device (see below)

**NB** : google may not always grant you access to a GPU. If you use it a lot, google will sometimes ask you to wait a few hours / a day before being able to use a GPU again. So it is better to test everything works on CPU on small data, and then run on GPU on full data.

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
#from tqdm import tqdm
from tqdm.notebook import tqdm # for progress bars in notebooks
from random import shuffle
import os
import pandas as pd


In [6]:
# in order to use a GPU
# modify notebook settings:
# Edit > Notebook settings > Hardware accelerator => select "GPU"

# if a GPU is available, we will use it
if torch.cuda.is_available():
    # objet torch.device
    DEVICE = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())
    device_id = torch.cuda.current_device()
    gpu_properties = torch.cuda.get_device_properties(device_id)
    print("We will use GPU %d (%s) of compute capability %d.%d with "
          "%.2fGb total memory.\n" %
          (device_id,
          gpu_properties.name,
          gpu_properties.major,
          gpu_properties.minor,
          gpu_properties.total_memory / 1e9))

else:
    print('No GPU available, using the CPU instead.')
    DEVICE = torch.device("cpu")



No GPU available, using the CPU instead.


## DATA

We use a French sentiment analysis dataset (subset of the French part of the multilingual dataset CLS <a href="https://aclanthology.org/P10-1114/">(Prettenhofer and Stein, 2010)</a>, included in the FLUE benchmark <a href="https://aclanthology.org/2020.lrec-1.302/">(Le et al., 2020)</a>, available at https://github.com/getalp/Flaubert/tree/master/flue.

Sentiment analysis is a document classification task. The documents are book/music/movie reviews. Gold labels have been mapped to 0 negative and 1 for positive reviews (neutral reviews have been ignored).

We provide a train / validation split.


## Loading data

In [7]:
def load_cls_file(file_path):
    reviews, labels = [], []
    with open(file_path) as f:
        line = f.readline()
        while line:
            review, label = line.strip().split('\t')
            reviews.append(review.strip("\""))
            labels.append(int(label))
            line = f.readline()
    return reviews, labels

def load_cls_dataset(file_path, section='books'):
    part2X, part2y = {}, {}
    for part in ['train', 'valid']:
        part2X[part], part2y[part] = load_cls_file(os.path.join(file_path, section, part+'_0.tsv'))
    return part2X, part2y

In [8]:
data_dir = './data_flue_cls_fr/'

if not os.path.exists(data_dir):
  # shell commands can be run using !
  !pip install wget
  import wget

  # The URL for the dataset zip file.
  url = 'http://www.linguist.univ-paris-diderot.fr/~mcandito/divers/data_flue_cls_fr.tgz'


  if not os.path.exists('./data_flue_cls_fr.tgz'):
    print('Downloading dataset')
    wget.download(url, './data_flue_cls_fr.tgz')
    !tar zxf data_flue_cls_fr.tgz



In [9]:
X = {} # key1=cat, key2=part, val= list of reviews
y = {} # key1=cat, key2=part, val= list of labels
y_train = [] # les labels "gold" pour la partie train
y_valid = [] # les labels "gold" pour la partie validation
for genre in ['books', 'dvd', 'music']:
    X[genre], y[genre] = load_cls_dataset(data_dir, genre)
    y_train += y[genre]['train']
    y_valid += y[genre]['valid']

In [10]:
df = {} # key=cat, val = data frame
for part in ['train', 'valid']:
    df[part] = pd.DataFrame.from_dict(
    {'review': X['books'][part] + X['dvd'][part] + X['music'][part],
     'label': y['books'][part] + y['dvd'][part] + y['music'][part],
     'genre': ['books' for _ in range(len(X['books'][part]))] \
             + ['dvd' for _ in range(len(X['dvd'][part]))] \
             + ['music' for _ in range(len(X['music'][part]))]})

id2label = ['Négatif', 'Positif']

In [11]:
pd.options.display.max_colwidth = 300
df['train'].head(10)

,review,label,genre
0,"Je voulais mettre 0 étoile mais c'est pas possible... Commençons par le positif (c'est rapide): ça parle de tout : muxle, endurance, souplesse, alimentation, échauffement mais ... mal, désespérement mal. L'auteur prétend avoir fait 15 ans de recherche avant de pondre son opus majus. Ben faudrait...",0,books
1,"Le récit de vie d'une femme américaine, avec ses contradictions, ses rêves de liberté, de justice (dans l'amérique de la guerre du Vietnam). Un récit qui n'est pas prêt de me quitter sur la complexité de la nature humaine dans ce qu'elle a de plus tortueux et de plus insaisissable. Un récit cons...",1,books
2,"Quel chef d'oeuvre que cette 'autobiographie' ! C'est à croire que la vie de Michaël Crichton est aussi palpitante, voire plus par moments, que celle des personnages de ses romans... Que d'aventures, de découvertes, d'expérience ! On comprend mieux, après la lecture de cet ouvrage, d'où lui vien...",1,books
3,"Si vous cherchez un livre simple d'initiation à Scheme pour comprendre et modifier les Script-Fu de Gimp, ne faites pas comme moi, choisissez-en un autre. Si vous voulez étudier ce langage d'un point de vue théorique, sans allumer votre ordinateur, en entrant dans le détail des algorithmes, sans...",1,books
4,Ce livre est très intéressant à lire à plus d'un titre. Certes JF Revel voit juste dans son analyse au sujet de la pensée ambiante vis-à-vis des USA. Cependant il manque totalement de rigueur scientifique (beaucoup d'assertions sans références bibliographiques). Cet ouvrage bien qu'éclairant ne ...,0,books
5,"J'avais déjà été déçue par """"Dossier Benton"""". Mais là, que dire de """"L'Ile des Chiens"""" ??? Quel ennui ! Les personnages sont burlesques, et l'intrigue initéressante ! Je suis pourtant une inconditionnelle de celle que je considérais comme LA reine du thriller. On ne ressuscitera pas Benton, ma...",0,books
6,"Michel Hoàng trace un portrait captivant du plus grand conquérant de l'histoire. Il dépeint les mœurs de ces nomades de la steppe qui engendreront Tèmudjin, le fils de petit chef qui s'élèvera au range de Khan océanique. L'ascension lente et patiente faite à coup d'alliances et « desalliance » d...",1,books
7,"Que lui avons-nous fait? Pourquoi la belle Amélie se croit-elle obligée de nous décevoir chaque année? Si le but est de nous prouver que le génie ne s'utilise pas comme une clé de 12, la démonstration en a été faite depuis longtemps.",0,books
8,"Il y a deux manières d'envisager ce livre. Sur le plan de la méthode, Encel expose le Golan d'un point de vue géopolitique, ce qui peut être pertinent... jusqu'à un certain point. En effet, au-delà de certaines imprécisions, voire confusions dans la réalité historique du Golan, l'auteur en vient...",0,books
9,"On s'attend avec ce livre à découvrir un pays, une époque, et surtout lire un roman d'espionnage, quelque chose de palpitant, qui s'emballe, un livre que l'on ne peut plus quitter quand on l'a ouvert. Malheureusement, le scénario s'essouffle au fil des pages, pour devenir carrément asthmatique à...",0,books


## TODO1 : Tokenizing the documents

In [12]:
import torch

# import (after potential installation) of the transformers module
try:
  from transformers import AutoModel, AutoTokenizer
except ImportError:
  !pip install transformers
  from transformers import AutoModel, AutoTokenizer

# flaubert's tokenization uses as first step a tokenization into words by moses
try:
  import sacremoses
except ImportError:
  !pip install sacremoses
  import sacremoses

# flaubert's tokenizer
tokenizer = AutoTokenizer.from_pretrained("flaubert/flaubert_base_cased")
print(type(tokenizer))

/home/fatoumata/Documents/NLP/TP4/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<class 'transformers.models.flaubert.tokenization_flaubert.FlaubertTokenizer'>


In [13]:
MAX_LENGTH=200

# tokenize the reviews with flaubert
# fill the itok_seqs dictionary
# for the keys 'train' and 'valid'
# with the token ids for the training / valid reviews
itok_seqs = {}
attention_masks = {}
for split in ['train', 'valid']:
    enc = tokenizer(
        df[split]['review'].tolist(),
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
        return_attention_mask=True
    )
    itok_seqs[split] = enc['input_ids']
    attention_masks[split] = enc['attention_mask']

print('train input_ids shape:', itok_seqs['train'].shape)
print('valid input_ids shape:', itok_seqs['valid'].shape)

train input_ids shape: torch.Size([4799, 200])
valid input_ids shape: torch.Size([1198, 200])


## TODO2 : build tensors for data

In [29]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

# TODO study TensorDataset
# and DataLoader

# and simply build a TensorDataset
# - for the training reviews
# - for the validation reviews

# the corresponding gold labels also need to be put into torch tensors

# TensorDataset is for dataset wrapping tensors
# Each sample will be retrieved by indexing tensors along the first dimension
dataset = {}
dataset['train'] = TensorDataset(itok_seqs['train'], attention_masks['train'], torch.tensor(y_train))
dataset['valid'] = TensorDataset(itok_seqs['valid'], attention_masks['valid'], torch.tensor(y_valid))

# note : the labels are already ids (0 or 1), if they weren't, we would need to convert them to ids first

BATCH_SIZE = 32

# DataLoader is convenient to easily make batche, and to choose how to iterate on batches
# (in random order for training, sequential order for validation cf. no need for random order)

train_dataloader = DataLoader(dataset['train'], sampler=RandomSampler(dataset['train']), batch_size=BATCH_SIZE)
valid_dataloader = DataLoader(dataset['valid'], sampler=SequentialSampler(dataset['valid']), batch_size=BATCH_SIZE)


## TODO3 : define the network for the document classifier



The choice between fine-tuning versus freezing the pretrained language model (PLM) can be made in the __init__, setting requires_grad to False to the whole PLM (see below).


In [30]:
class DocClassifier(nn.Module):

    def __init__(self, device, num_labels, plm_model, plm_embedding_dim, freeze_plm = False, hidden_size = 100, dropout=0.1):
        super(DocClassifier, self).__init__()

        self.freeze_plm = freeze_plm

        self.plm_layer = plm_model.to(device)

        # to freeze bert model parameters : don't compute their gradients
        if freeze_plm:
            for p in self.plm_layer.parameters():
                p.requires_grad = False
        print("Classifier instance, freeze_plm = %s" % str(freeze_plm))

        self.classifier = nn.Linear(plm_embedding_dim, hidden_size).to(device)
        self.classifier2 = nn.Linear(hidden_size, num_labels).to(device)
        self.dropout = nn.Dropout(dropout).to(device)

    def forward(self, itok_seqs):
        """
        Inputs: a batch of reviews = a batch of token id sequences

        Output: log_softmax scores for the whole batch (batch_size x num_labels)
        """

        # run the plm model on the sentences of the batch
        contextual_emb_seq = self.plm_layer(itok_seqs).last_hidden_state # batch_size, max_length, bert_emb_size

        # get the embedding of the [CLS] token, which is the first token
        cls_emb = contextual_emb_seq[:, 0, :]  # batch_size, emb_dim

        # pass through MLP
        x = self.dropout(cls_emb)
        x = F.relu(self.classifier(x))
        x = self.dropout(x)
        logits = self.classifier2(x)

        return F.log_softmax(logits, dim=1)

    def predict_on_dataset(self, dataloader):
        """
        Run classifier on dataloader and compute accuracy
        ** without computing gradients **
        Inputs = DataLoader instance
        Returns:
         - list of predicted label ids
         - list of gold labels
         - average loss
        """
        pred_labels = []
        gold_labels = []
        # VERY IMPORTANT : when evaluating the model, we need to toggle off train mode
        #                  so that dropout is off when running the bert model
        self.eval()
        data_loss = 0
        with torch.no_grad():
            for batch in dataloader:
                b_itok_seqs, b_attention_masks, b_gold_labels = batch
                b_itok_seqs = b_itok_seqs.to(DEVICE)
                b_attention_masks = b_attention_masks.to(DEVICE)
                b_gold_labels = b_gold_labels.to(DEVICE)

                logits = self.forward(b_itok_seqs)
                loss = F.nll_loss(logits, b_gold_labels)
                data_loss += loss.item()

                pred = torch.argmax(logits, dim=1)
                pred_labels.extend(pred.cpu().numpy())
                gold_labels.extend(b_gold_labels.cpu().numpy())

        return pred_labels, gold_labels, data_loss / len(dataloader)

    def evaluate(self, gold_labels, pred_labels):
        from sklearn.metrics import accuracy_score
        accuracy = accuracy_score(gold_labels, pred_labels)
        print(f"Accuracy: {accuracy}")
        return accuracy

## TODO4 : Training method


In [31]:
VERBOSE=False
NB_EPOCHS=5

def train_classifier(classifier, lr, train_dataloader, valid_dataloader):

  # the optimizer: we declare which parameters are to be learnt
  #optimizer = optim.SGD(classifier.parameters(), lr=lr)
  optimizer = optim.Adam(classifier.parameters(), lr=lr)

  prev_valid_loss = float('inf')

  for epoch in range(1, NB_EPOCHS + 1):
      print(f"Epoch {epoch}...")
      train_loss = 0

      # toggle on the training mode
      # (cf. certain modules behave differently (e.g. dropout is only in training mode)
      classifier.train()

      # IMPORTANT
      # we move the batches to the DEVICE
      for iteration, batch in enumerate(train_dataloader):
          b_itok_seqs, b_attention_masks, b_gold_labels = batch
          b_itok_seqs = b_itok_seqs.to(DEVICE, non_blocking=True)
          b_attention_masks = b_attention_masks.to(DEVICE, non_blocking=True)
          b_gold_labels = b_gold_labels.to(DEVICE, non_blocking=True)

          optimizer.zero_grad()
          logits = classifier(b_itok_seqs)
          loss = F.nll_loss(logits, b_gold_labels)
          loss.backward()
          optimizer.step()

          train_loss += loss.item()

      train_loss /= len(train_dataloader)

      # evaluate on validation
      pred_labels, gold_labels, valid_loss = classifier.predict_on_dataset(valid_dataloader)
      accuracy = classifier.evaluate(gold_labels, pred_labels)

      print(f"Epoch {epoch}: Train loss {train_loss:.4f}, Valid loss {valid_loss:.4f}, Valid acc {accuracy:.4f}")

      # early stopping
      if valid_loss > prev_valid_loss:
          print("Early stopping")
          break
      prev_valid_loss = valid_loss

In [32]:
from transformers import AutoModel, AutoTokenizer, AutoConfig

flaubert_model = AutoModel.from_pretrained("flaubert/flaubert_base_cased", return_dict=True)
flaubert_config = AutoConfig.from_pretrained("flaubert/flaubert_base_cased")

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 10968.87it/s]
FlaubertModel LOAD REPORT from: flaubert/flaubert_base_cased
Key                  | Status     |  | 
---------------------+------------+--+-
pred_layer.proj.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Let's train !

In [ ]:
NUM_LABELS = 2
DROPOUT=0.1
HIDDEN_SIZE=300 # the Flaubert paper provides tuned hyperparameters. They used a larger hidden size here

FREEZE_PLM = True
LR = 0.0005

# NB: when the PLM is not frozen
# the learning rate must generally be smaller
#FREEZE_PLM = False
#LR = 0.000005

plm_embedding_dim = flaubert_config.emb_dim

classifier = DocClassifier(DEVICE, NUM_LABELS, flaubert_model, plm_embedding_dim, freeze_plm=FREEZE_PLM, hidden_size=HIDDEN_SIZE, dropout=DROPOUT)

# we move all the parameters' tensors to the selected device
classifier = classifier.to(DEVICE)

# for full training on GPU
print("Training on full dataset...")
train_classifier(classifier, LR, train_dataloader, valid_dataloader)

#evaluate on validation set
pred_labels, gold_labels, _ = classifier.predict_on_dataset(valid_dataloader)
accuracy = classifier.evaluate(gold_labels, pred_labels)
print(f"Final validation accuracy: {accuracy}")

Classifier instance, freeze_plm = True
Testing on small dataset...
Epoch 1...
Accuracy: 0.46
Epoch 1: Train loss 0.6994, Valid loss 0.7030, Valid acc 0.4600
Epoch 2...
Accuracy: 0.64
Epoch 2: Train loss 0.6044, Valid loss 0.7429, Valid acc 0.6400
Early stopping
Accuracy: 0.64
Final small validation accuracy: 0.64
